In [1]:
import os
import json
import glob
import requests
import pdfplumber
import pandas as pd

In [ ]:
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")          # API ключ для доступа к Mistral
PDF_FILES = "./pdf_files"                               # Папка с PDF файлами для обработки 
OUTPUT_FILE = "report.xlsx"                             # Имя выходного Excel файла
MODEL = "mistral-small-latest"                          # Модель Mistral для обработки текста
MAX_TOKENS = 5000                                       # Максимальное количество токенов для обработки каждым запросом

In [ ]:
# Инструкция с действиями для модели
SYSTEM_PROMPT = """Ты — аналитик, классифицирующий PDF-отчёты.
Для каждого отчёта верни JSON-объект СТРОГО в следующем формате (без markdown, без ```):

{
  "author_title": "Автор отчёта; Название отчёта",
  "topic": "Тематика одним словом",
  "keywords": "ключевое1; ключевое2; ... (примерно 10 штук, через точку с запятой)",
  "summary": "Краткий итог в 3 - 5 предложениях."
}

Правила:
- Автор — организация или лицо, выпустившее отчёт. Если не указан явно — «Не указан».
- Тематика — одно слово (Здравоохранение, Финансы, Технологии, Маркетинг и т.д.).
- Ключевые слова — примерно 10, через точку с запятой.
- Summary — 3 - 5  предложения, основная суть отчёта, если в конце текста есть фраза "Длинна текста слишком большая. Файл был обрезан"
  ОБЯЗАТЕЛЬНО в конце summary добавь фразу "Отчет был создан не для всего файла".
- Верни ТОЛЬКО валидный JSON, без пояснений."""

In [4]:
def extract_text_from_file(pdf_path: str) -> str:
    """Извлекает текст из PDF."""
    text_parts = []
    # Открываем файл и считываем текст с помощью pdfplumber 
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text_parts.append(page_text)

    # Объединяем весь текст в одну строку для подачи в модель
    full_text = "\n".join(text_parts)

    # Обрезаем файл до первых n слов чтобы ускорить ответ модели 
    if len(full_text.split()) > MAX_TOKENS:
        filename = os.path.basename(pdf_path)
        print(f"Длинна текста слишком большая. Файл {filename} был обрезан")
        full_text = "\n".join(full_text.split()[:MAX_TOKENS]) + " \nДлинна текста слишком большая. Файл был обрезан"
    return full_text

In [5]:
def classify_pdf(text: str, filename: str) -> dict:
    """Отправляет текст в Mistral для получения отчета"""
    # Промпт для модели с текстом файла 
    user_message = f"Файл: {filename}\nТекст отчёта:\n{text}"

    # Запрос к Mistral с инструкцией что делать (system prompt)
    # и текстом из файла для создания отчета
    resp = requests.post(
        url="https://api.mistral.ai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {MISTRAL_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": MODEL,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message},
            ],
            "temperature": 0.2,
        },
    )
    # Пытаемся получить ответ модели в почти json формате
    try:
        raw = resp.json()["choices"][0]["message"]["content"]
    except KeyError:
        print("Не удалось получить ответ от модели. Ошибка {resp}")
        return {
            "author_title": "Ошибка модели",
            "topic": "—",
            "keywords": "—",
            "summary": raw[:300],
            }
    
    # Обработка ответа для привидения его к json формату
    raw = raw.strip()

    if raw.startswith("```"):
        raw = raw.strip("```")
    raw = raw.strip()
    if raw.startswith("json"):
        raw = raw[4:].strip()
    print(raw)

    # Парсим финальный json для отчета
    try:
        data = json.loads(raw)
        print(f"Отчет по файлу {filename} успешно создан")
        return data
    except json.JSONDecodeError:
        print(f"Не удалось распарсить JSON. Отчет по файлу {filename} не создан")
        return {
            "author_title": "Ошибка парсинга",
            "topic": "—",
            "keywords": "—",
            "summary": raw[:300],
            }



In [6]:
def main():
    """Главная функция для обработки файлов и создания финального отчета"""
    # Находим все PDF-файлы в папке
    pdf_files = sorted(glob.glob(os.path.join(PDF_FILES, "*.pdf")))
    if not pdf_files:
        print(f"PDF-файлы не найдены в папке '{PDF_FILES}'.")
        return

    print(f"Найдено PDF: {len(pdf_files)}")
    print(f"Модель: {MODEL}")
    results = []
    # Обрабатываем все файлы 
    for i, pdf_path in enumerate(pdf_files, 1):
        filename = os.path.basename(pdf_path)
        print(f"[{i}/{len(pdf_files)}] Обработка: {filename}")

        text = extract_text_from_file(pdf_path)
        if not text:
            print("Текст не извлечён. Файл {filename} пропущен.")
            results.append({
                "Автор и название": f"(не извлечён)",
                "Тематика": "—",
                "Ключевые слова": "—",
                "Краткое summary": "Текст не удалось извлечь.",
            })
            continue
        # Создаем отчет для каждого файла 
        data = classify_pdf(text, filename)
        results.append({
            "Автор и название": data.get("author_title", "—"),
            "Тематика": data.get("topic", "—"),
            "Ключевые слова": data.get("keywords", "—"),
            "Краткое summary": data.get("summary", "—"),
        })

    df = pd.DataFrame(results)

    df.to_excel(OUTPUT_FILE, index=False, engine="openpyxl")
    print(f"Результаты сохранены в {OUTPUT_FILE}")

In [7]:
if __name__ == "__main__":
    main()

Найдено PDF: 2
Модель: mistral-small-latest
[1/2] Обработка: DSM.pdf
Длинна текста слишком большая. Файл DSM.pdf был обрезан
{"author_title": "DSM Group; Фармацевтический рынок России. Май 2025 — розничный аудит фармацевтического рынка РФ", "topic": "Здравоохранение", "keywords": "фармацевтический рынок; лекарственные препараты; розничные продажи; аптечный рынок; локализованные препараты; импортные препараты; рецептурные препараты; безрецептурные препараты; биологически активные добавки; АТС-группы; производители лекарств; дженерики; оригинальные препараты; ценовые сегменты; фармацевтическая статистика; DSM Group; ISO 9001:2015", "summary": "Отчёт DSM Group представляет анализ розничного фармацевтического рынка России за май 2025 года. Объём коммерческого рынка лекарственных препаратов составил 146 млрд рублей, что на 15,1% больше, чем в мае 2024 года. В натуральном выражении реализовано 349,2 млн упаковок, средняя стоимость упаковки — 418,2 рубля. Доля препаратов дороже 1000 рублей ув